## Preprocesamiento

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/creditrisk360_eda.parquet")
print(df.shape)
df.head(3)

(549566, 40)


,id,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,total_rec_int,recoveries,collection_recovery_fee,last_pymnt_amnt,application_type,mort_acc,pub_rec_bankruptcies,default,fico_tramo,issue_year
0,68407277,3600.0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,...,821.72,0.0,0.0,122.67,Individual,1.0,0.0,0,660-680,2015
1,68476668,20000.0,20000.0,36 months,9.17,637.58,B,B2,10+ years,MORTGAGE,...,1393.80,0.0,0.0,15681.05,Individual,4.0,0.0,0,660-680,2015
2,67275481,20000.0,20000.0,36 months,8.49,631.26,B,B1,10+ years,MORTGAGE,...,1538.51,0.0,0.0,14618.23,Individual,3.0,0.0,0,700-720,2015


In [2]:
TARGET = "default"

FEATURES_ORDINAL = ["grade", "sub_grade", "term", "emp_length"]
FEATURES_NUMERIC = [
    "fico_range_low", "int_rate", "dti", "revol_util",
    "annual_inc", "open_acc", "delinq_2yrs", "inq_last_6mths",
    "pub_rec", "mort_acc", "pub_rec_bankruptcies",
    "mths_since_last_delinq", "never_delinq"  
]
FEATURES_NOMINAL = ["purpose", "home_ownership", "verification_status"]

In [3]:
# emp_length: NaN = no informado, categoría propia antes del map
df["emp_length"] = df["emp_length"].fillna("unknown")

# mort_acc: sin hipotecas conocidas = 0
df["mort_acc"] = df["mort_acc"].fillna(0)

# pub_rec_bankruptcies: sin registros = 0
df["pub_rec_bankruptcies"] = df["pub_rec_bankruptcies"].fillna(0)

# mths_since_last_delinq: NaN = nunca delinquió (mejor perfil)
# Imputar con valor alto (999) + flag binario
df["mths_since_last_delinq"] = df["mths_since_last_delinq"].fillna(999)
df["never_delinq"] = (df["mths_since_last_delinq"] == 999).astype(int)

# Numéricas con pocos NaNs: mediana
for col in ["dti", "revol_util", "annual_inc", "open_acc",
            "delinq_2yrs", "inq_last_6mths", "pub_rec"]:
    df[col] = df[col].fillna(df[col].median())

# Verificación
print(df[FEATURES_ORDINAL + FEATURES_NUMERIC].isnull().sum().sum(), "NaNs restantes")

0 NaNs restantes


In [4]:
grade_order    = list("ABCDEFG")
subgrade_order = [f"{g}{n}" for g in grade_order for n in range(1, 6)]
term_order     = [" 36 months", " 60 months"]
emp_order = ["< 1 year","1 year","2 years","3 years","4 years",
             "5 years","6 years","7 years","8 years","9 years","10+ years","unknown"]

df["grade_enc"]      = df["grade"].map({g: i for i, g in enumerate(grade_order)})
df["sub_grade_enc"]  = df["sub_grade"].map({sg: i for i, sg in enumerate(subgrade_order)})
df["term_enc"]       = df["term"].map({t: i for i, t in enumerate(term_order)})
df["emp_length_enc"] = df["emp_length"].map({e: i for i, e in enumerate(emp_order)})

df[["grade","grade_enc","sub_grade","sub_grade_enc"]].drop_duplicates().sort_values("grade_enc").head(10)

,grade,grade_enc,sub_grade,sub_grade_enc
3,A,0,A2,1
76,A,0,A3,2
6,A,0,A4,3
44,A,0,A5,4
11,A,0,A1,0
1,B,1,B2,6
2,B,1,B1,5
31,B,1,B5,9
12,B,1,B4,8
14,B,1,B3,7


In [5]:
df["log_annual_inc"] = np.log1p(df["annual_inc"])
df["log_dti"]        = np.log1p(df["dti"].clip(lower=0))
df["revol_util"]     = df["revol_util"].clip(0, 100)

df[["annual_inc","log_annual_inc","dti","log_dti"]].skew().round(2)

annual_inc        46.22
log_annual_inc    -1.99
dti               27.94
log_dti           -1.13
dtype: float64

In [6]:
df = pd.get_dummies(df, columns=FEATURES_NOMINAL, drop_first=True, dtype=int)

dummy_cols = [c for c in df.columns if c.startswith(("purpose_","home_ownership_","verification_status_"))]
print(f"Dummies generadas: {len(dummy_cols)}")
print(dummy_cols)

Dummies generadas: 20
['purpose_credit_card', 'purpose_debt_consolidation', 'purpose_educational', 'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase', 'purpose_medical', 'purpose_moving', 'purpose_other', 'purpose_renewable_energy', 'purpose_small_business', 'purpose_vacation', 'purpose_wedding', 'home_ownership_MORTGAGE', 'home_ownership_NONE', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'verification_status_Source Verified', 'verification_status_Verified']


In [7]:
cutoff_year = 2016

train = df[df["issue_year"] <= cutoff_year].copy()
test  = df[df["issue_year"] >  cutoff_year].copy()

print(f"Train : {len(train):>7,} rows | default rate: {train[TARGET].mean():.1%}")
print(f"Test  : {len(test):>7,} rows  | default rate: {test[TARGET].mean():.1%}")
print(f"Ratio : {len(train)/len(df):.0%} train / {len(test)/len(df):.0%} test")

Train : 451,805 rows | default rate: 20.2%
Test  :  97,761 rows  | default rate: 27.2%
Ratio : 82% train / 18% test


In [8]:
# Bins de fico (señal no lineal)
df["fico_bin"] = pd.cut(df["fico_range_low"],
                         bins=[0, 650, 670, 700, 720, 750, 850, 999],
                         labels=[0, 1, 2, 3, 4, 5, 6]).astype(int)

# Interaction term grade x dti
df["grade_dti"] = df["grade_enc"] * df["log_dti"]

# Recalcular train/test con las nuevas columnas
train = df[df["issue_year"] <= cutoff_year].copy()
test  = df[df["issue_year"] >  cutoff_year].copy()

ENCODED_FEATURES = (
    ["grade_enc", "sub_grade_enc", "term_enc", "emp_length_enc",
     "fico_range_low", "int_rate", "log_dti", "revol_util",
     "log_annual_inc", "open_acc", "delinq_2yrs", "inq_last_6mths",
     "pub_rec", "mort_acc", "pub_rec_bankruptcies",
     "mths_since_last_delinq", "never_delinq",
     "fico_bin", "grade_dti"]
    + dummy_cols
)

X_train, y_train = train[ENCODED_FEATURES], train[TARGET]
X_test,  y_test  = test[ENCODED_FEATURES],  test[TARGET]

print("NaNs en X_train:", X_train.isnull().sum().sum())
print("NaNs en X_test :", X_test.isnull().sum().sum())
print(f"Features totales : {len(ENCODED_FEATURES)}")
print(f"X_train shape    : {X_train.shape}")
print(f"X_test  shape    : {X_test.shape}")

train[ENCODED_FEATURES + [TARGET]].to_parquet("../data/train_processed.parquet", index=False)
test[ENCODED_FEATURES + [TARGET]].to_parquet("../data/test_processed.parquet",  index=False)
print("\nParquets guardados ✓")

NaNs en X_train: 0
NaNs en X_test : 0
Features totales : 39
X_train shape    : (451805, 39)
X_test  shape    : (97761, 39)

Parquets guardados ✓
